# Benchmark multi-sources étendu — Collection réelle

Objectif :

- partir d’un export réel de collection ;
- sélectionner un échantillon représentatif par série ;
- utiliser le premier volume disponible par série ;
- limiter les doublons liés aux séries longues ;
- benchmarker Google Books, OpenLibrary, BNF et Nudger sur cet échantillon.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))



from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)
from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)

from collectionlens.api.open_library_api import (
    search_book_by_isbn as openlibrary_isbn,
)

from collectionlens.api.bnf_api import (
    search_book_by_isbn as bnf_isbn,
)

from collectionlens.api.nudger_dataset import (
    load_nudger_dataset,
    build_nudger_search_function,
)

from collectionlens.benchmark.api_benchmark import (
    benchmark_source_in_batches,
)



## 1. Construction du dataset benchmark

In [2]:
collection_path = PROJECT_ROOT / "data" / "raw" / "collection_export.csv"

df_collection = pd.read_csv(
    collection_path,
    dtype={
        "EAN": str,
        "isbn_clean": str,
    },
    sep=";",
    low_memory=False,
)

df_collection.head()

,Titre de la série,Type,Collection,Catégory,Titre de l'album,EAN,Tome,Date de publication,Editeur,Auteurs,...,Version numérique,A vendre,Numérotation,Prix d'achat,Cote,Etat,Ma note de l'album,Mon avis de l'album,Ma note de la série,Mon avis de la série
0,103ème Escadrille de Chasse,NaN,NaN,Mangas,NaN,9782888906063,NaN,2013-09-21T00:00:00.000Z,Paquet,Takizawa Seiho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001 Nights Stories,NaN,NaN,Mangas,2001 Nights Stories,9782344057001,1.0,2023-10-04T00:00:00.000Z,Glénat,Yukinobu Hoshino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2001 Nights Stories,NaN,NaN,Mangas,2001 Nights Stories,9782344057247,2.0,2023-10-04T00:00:00.000Z,Glénat,Yukinobu Hoshino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20th Century Boys,album simple N&B,NaN,Mangas,NaN,9782845380790,1.0,2003-12-22T00:00:00.000Z,Panini,Naoki Urasawa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20th Century Boys,album simple N&B,NaN,Mangas,NaN,9782845380998,2.0,2003-12-22T00:00:00.000Z,Panini,Naoki Urasawa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_collection_clean = df_collection.copy()

df_collection_clean = df_collection_clean[
    df_collection_clean["EAN"].notna()
].copy()

df_collection_clean["Tome"] = pd.to_numeric(
    df_collection_clean["Tome"],
    errors="coerce",
)

df_collection_clean["Titre de la série"] = (
    df_collection_clean["Titre de la série"]
    .fillna("")
    .str.strip()
)

In [4]:
df_first_volume_by_series = (
    df_collection_clean
    .sort_values(
        by=[
            "Titre de la série",
            "Tome",
            "Date de publication",
        ],
        ascending=True,
    )
    .groupby("Titre de la série", as_index=False)
    .first()
)

df_first_volume_by_series[
    [
        "Titre de la série",
        "Type",
        "Catégory",
        "Titre de l'album",
        "Tome",
        "EAN",
        "Editeur",
        "Auteurs",
    ]
].head()

,Titre de la série,Type,Catégory,Titre de l'album,Tome,EAN,Editeur,Auteurs
0,#les memes,album simple,BD,Chronique des âges farouches,1.0,9791038202733,Fluide Glacial,Sylvain Frécon
1,103ème Escadrille de Chasse,None,Mangas,None,NaN,9782888906063,Paquet,Takizawa Seiho
2,2001 Nights Stories,None,Mangas,2001 Nights Stories,1.0,9782344057001,Glénat,Yukinobu Hoshino
3,20th Century Boys,album simple N&B,Mangas,None,1.0,9782845380790,Panini,Naoki Urasawa
4,20th Century Boys - Spin off,None,Mangas,None,NaN,9782809425659,Panini,Naoki Urasawa


In [5]:
print("Nombre d'albums dans l'export :", len(df_collection))
print("Nombre d'albums avec ISBN :", len(df_collection_clean))
print("Nombre de séries représentées :", len(df_first_volume_by_series))

Nombre d'albums dans l'export : 5651
Nombre d'albums avec ISBN : 5651
Nombre de séries représentées : 1036


## 2. Vérification du dataset benchmark

In [6]:
df_first_volume_by_series["Type"].value_counts(dropna=False)

Type
album simple              361
None                      354
album simple N&B          259
intégrale complète         26
hors série                  7
Autre                       6
intégrale 2T                5
intégrale 3T                4
intégrale complète N&B      3
Fascicule / Softcover       2
intégrale 2T N&B            2
coffret intégrale           2
intégrale 4T                2
coffret complet             1
coffret intégrale N&B       1
intégrale 5T                1
Name: count, dtype: int64

In [7]:
df_first_volume_by_series["Catégory"].value_counts(dropna=False)

Catégory
Mangas    759
Comics    169
BD        104
None        4
Name: count, dtype: int64

In [8]:
(
    df_first_volume_by_series
    .groupby("Catégory")["Type"]
    .apply(lambda x: x.isna().sum())
    .reset_index(name="missing_type")
)

,Catégory,missing_type
0,BD,26
1,Comics,22
2,Mangas,303


In [9]:
df_first_volume_by_series[
    df_first_volume_by_series["Type"].isna()
][
    [
        "Titre de la série",
        "Catégory",
        "Titre de l'album",
        "Tome",
    ]
].head(20)

,Titre de la série,Catégory,Titre de l'album,Tome
1,103ème Escadrille de Chasse,Mangas,None,NaN
2,2001 Nights Stories,Mangas,2001 Nights Stories,1.0
4,20th Century Boys - Spin off,Mangas,None,NaN
6,24 Histoires d'un Temps Lointain,Mangas,None,NaN
7,25 Histoires d'un Monde en 4 Dimensions,Mangas,None,NaN
9,3 Rue des Mystères et Autres Histoires,Mangas,None,1.0
12,7 Shakespeares,Mangas,None,1.0
13,8 Man Infinity,Mangas,8 Man Infinity,1.0
16,Ace Attorney - Investigations,Mangas,None,1.0
17,Ace Attorney - Phoenix Wright,Mangas,None,1.0


In [10]:
output_dir = PROJECT_ROOT / "data" / "processed" / "extended_benchmark"
output_dir.mkdir(parents=True, exist_ok=True)

benchmark_input_path = output_dir / "benchmark_first_volume_by_series.csv"

df_first_volume_by_series.to_csv(
    benchmark_input_path,
    index=False,
)

benchmark_input_path

WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/processed/extended_benchmark/benchmark_first_volume_by_series.csv')

In [11]:
isbns = (
    df_first_volume_by_series["EAN"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

len(isbns)

1036

L'analyse des valeurs manquantes du champ `Type` montre qu'elles concernent principalement des mangas. Une vérification manuelle indique qu'il s'agit majoritairement de one-shots, d'éditions uniques ou de séries sans numérotation explicite.

Ces valeurs ne sont donc pas considérées comme des anomalies et ne nécessitent pas de correction pour la suite du benchmark.

Le benchmark étendu est construit à partir de l’export réel de collection.

Pour éviter de surreprésenter les séries longues, un seul volume est conservé par série : le premier tome disponible selon le numéro de tome et la date de publication.

Cette stratégie permet d’obtenir un échantillon plus représentatif de la diversité des séries présentes dans la collection.

## 3. Benchmark Google Books

In [13]:
from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)

df_google_books = benchmark_source_in_batches(
    source_name="google_books",
    search_func=google_books_isbn,
    isbns=isbns,
    batch_size=50,
    output_dir=output_dir / "google_books",
    sleep_seconds=1.0,
)

df_google_books.to_csv(
    output_dir / "google_books_results.csv",
    index=False,
)

df_google_books.head()

Traitement batch 1/21 (50 ISBN)
Traitement batch 2/21 (50 ISBN)
Traitement batch 3/21 (50 ISBN)
Traitement batch 4/21 (50 ISBN)
Traitement batch 5/21 (50 ISBN)
Traitement batch 6/21 (50 ISBN)
Traitement batch 7/21 (50 ISBN)
Traitement batch 8/21 (50 ISBN)
Traitement batch 9/21 (50 ISBN)
Traitement batch 10/21 (50 ISBN)
Traitement batch 11/21 (50 ISBN)
Traitement batch 12/21 (50 ISBN)
Traitement batch 13/21 (50 ISBN)
Traitement batch 14/21 (50 ISBN)
Traitement batch 15/21 (50 ISBN)
Traitement batch 16/21 (50 ISBN)
Traitement batch 17/21 (50 ISBN)
Traitement batch 18/21 (50 ISBN)
Traitement batch 19/21 (50 ISBN)
Traitement batch 20/21 (50 ISBN)
Traitement batch 21/21 (36 ISBN)


,source,source_id,google_books_id,isbn,title,subtitle,authors,publisher,published_date,language,...,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,raw_data,found,error,status_code,isbn_query
0,google_books,ZXHozgEACAAJ,ZXHozgEACAAJ,9791038202733,#LesMémés Tome 1,None,[Frecon],None,2022-02-02,fr,...,NaN,http://books.google.fr/books?id=ZXHozgEACAAJ&d...,http://books.google.fr/books?id=ZXHozgEACAAJ&d...,https://books.google.com/books/about/LesM%C3%A...,"[{'type': 'ISBN_13', 'identifier': '9791038202...","{'kind': 'books#volume', 'id': 'ZXHozgEACAAJ',...",True,None,200,9791038202733
1,google_books,y4U2ngEACAAJ,y4U2ngEACAAJ,9782888906063,103e escadrille de chasse,None,[Seiho Takizawa],None,2013,fr,...,NaN,http://books.google.fr/books?id=y4U2ngEACAAJ&d...,http://books.google.fr/books?id=y4U2ngEACAAJ&d...,https://books.google.com/books/about/103e_esca...,"[{'type': 'ISBN_10', 'identifier': '2888906066...","{'kind': 'books#volume', 'id': 'y4U2ngEACAAJ',...",True,None,200,9782888906063
2,google_books,XIAp0AEACAAJ,XIAp0AEACAAJ,9782344057001,2001 Nights Stories Tome 1,None,[],None,2023-10-04,fr,...,NaN,http://books.google.fr/books?id=XIAp0AEACAAJ&d...,http://books.google.fr/books?id=XIAp0AEACAAJ&d...,https://books.google.com/books/about/2001_Nigh...,"[{'type': 'ISBN_10', 'identifier': '2344057005...","{'kind': 'books#volume', 'id': 'XIAp0AEACAAJ',...",True,None,200,9782344057001
3,google_books,M8IvNAAACAAJ,M8IvNAAACAAJ,9782845380790,20th century boys,None,[Naoki Urasawa],None,2010,fr,...,NaN,http://books.google.fr/books?id=M8IvNAAACAAJ&d...,http://books.google.fr/books?id=M8IvNAAACAAJ&d...,https://books.google.com/books/about/20th_cent...,"[{'type': 'ISBN_10', 'identifier': '2845380798...","{'kind': 'books#volume', 'id': 'M8IvNAAACAAJ',...",True,None,200,9782845380790
4,google_books,0D2jMQEACAAJ,0D2jMQEACAAJ,9782809425659,20th century boys,spin off : recueil d'histoires d'Ujiko Ujio,[Naoki Urasawa],None,2012,fr,...,NaN,http://books.google.fr/books?id=0D2jMQEACAAJ&d...,http://books.google.fr/books?id=0D2jMQEACAAJ&d...,https://books.google.com/books/about/20th_cent...,"[{'type': 'ISBN_10', 'identifier': '2809425655...","{'kind': 'books#volume', 'id': '0D2jMQEACAAJ',...",True,None,200,9782809425659


## 4. Benchmark open library

In [14]:
from collectionlens.api.open_library_api import (
    search_book_by_isbn as openlibrary_isbn,
)

df_openlibrary = benchmark_source_in_batches(
    source_name="openlibrary",
    search_func=openlibrary_isbn,
    isbns=isbns,
    batch_size=50,
    output_dir=output_dir / "openlibrary",
    sleep_seconds=1.0,
)

df_openlibrary.to_csv(
    output_dir / "openlibrary_results.csv",
    index=False,
)

df_openlibrary.head()

Traitement batch 1/21 (50 ISBN)
Traitement batch 2/21 (50 ISBN)
Traitement batch 3/21 (50 ISBN)
Traitement batch 4/21 (50 ISBN)
Traitement batch 5/21 (50 ISBN)
Traitement batch 6/21 (50 ISBN)
Traitement batch 7/21 (50 ISBN)
Traitement batch 8/21 (50 ISBN)
Traitement batch 9/21 (50 ISBN)
Traitement batch 10/21 (50 ISBN)
Traitement batch 11/21 (50 ISBN)
Traitement batch 12/21 (50 ISBN)
Traitement batch 13/21 (50 ISBN)
Traitement batch 14/21 (50 ISBN)
Traitement batch 15/21 (50 ISBN)
Traitement batch 16/21 (50 ISBN)
Traitement batch 17/21 (50 ISBN)
Traitement batch 18/21 (50 ISBN)
Traitement batch 19/21 (50 ISBN)
Traitement batch 20/21 (50 ISBN)
Traitement batch 21/21 (36 ISBN)


,source,isbn_query,found,error,status_code,source_id,openlibrary_key,isbn,title,subtitle,...,page_count,print_type,maturity_rating,average_rating,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,raw_data
0,openlibrary,9791038202733,False,no_result,200.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,openlibrary,9782888906063,False,no_result,200.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,openlibrary,9782344057001,False,no_result,200.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,openlibrary,9782845380790,False,no_result,200.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,openlibrary,9782809425659,False,no_result,200.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Benchmark BNF

In [15]:
from collectionlens.api.bnf_api import (
    search_book_by_isbn as bnf_isbn,
)

df_bnf = benchmark_source_in_batches(
    source_name="bnf",
    search_func=bnf_isbn,
    isbns=isbns,
    batch_size=50,
    output_dir=output_dir / "bnf",
    sleep_seconds=1.0,
)

df_bnf.to_csv(
    output_dir / "bnf_results.csv",
    index=False,
)

df_bnf.head()

Traitement batch 1/21 (50 ISBN)
Traitement batch 2/21 (50 ISBN)
Traitement batch 3/21 (50 ISBN)
Traitement batch 4/21 (50 ISBN)
Traitement batch 5/21 (50 ISBN)
Traitement batch 6/21 (50 ISBN)
Traitement batch 7/21 (50 ISBN)
Traitement batch 8/21 (50 ISBN)
Traitement batch 9/21 (50 ISBN)
Traitement batch 10/21 (50 ISBN)
Traitement batch 11/21 (50 ISBN)
Traitement batch 12/21 (50 ISBN)
Traitement batch 13/21 (50 ISBN)
Traitement batch 14/21 (50 ISBN)
Traitement batch 15/21 (50 ISBN)
Traitement batch 16/21 (50 ISBN)
Traitement batch 17/21 (50 ISBN)
Traitement batch 18/21 (50 ISBN)
Traitement batch 19/21 (50 ISBN)
Traitement batch 20/21 (50 ISBN)
Traitement batch 21/21 (36 ISBN)


,source,isbn_query,found,error,status_code,source_id,bnf_ark,isbn,title,subtitle,...,average_rating,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,format,types,rights,raw_data
0,bnf,9791038202733,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb46991952p,http://catalogue.bnf.fr/ark:/12148/cb46991952p,9791038202733,"Chroniques des âges farouches / scénario, dess...",NaN,...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb46991952...,1 vol. (53 p.) : ill. en coul. ; 30 cm,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
1,bnf,9782888906063,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb43489163k,http://catalogue.bnf.fr/ark:/12148/cb43489163k,9782888906063,103e escadrille de chasse / Takizawa Seiho,NaN,...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb43489163...,1 vol. (224 p.) : illustrations en noir et bla...,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
2,bnf,9782344057001,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb47348636c,http://catalogue.bnf.fr/ark:/12148/cb47348636c,9782344057001,2001 nights stories. 1 : version d'origine / Y...,NaN,...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb47348636...,1 vol. (336 p.) : ill. en coul. ; 31 cm,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
3,bnf,9782845380790,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bnf,9782809425659,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb42742695f,http://catalogue.bnf.fr/ark:/12148/cb42742695f,9782809425659,20th century boys : spin off : recueil d'histo...,NaN,...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb42742695...,"1 vol. (non paginé [176] p.) : ill., couv. ill...","[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."


## 6. Benchmark nudger

In [16]:
nudger_path = PROJECT_ROOT / "data" / "external" / "nudger" / "open4goods-isbn-dataset.csv"

df_nudger_source = load_nudger_dataset(
    csv_path=nudger_path,
    isbn_column="isbn",
)

nudger_isbn = build_nudger_search_function(df_nudger_source)


df_nudger = benchmark_source_in_batches(
    source_name="nudger",
    search_func=nudger_isbn,
    isbns=isbns,
    batch_size=100,
    output_dir=output_dir / "nudger",
    sleep_seconds=0.0,
)

df_nudger.to_csv(
    output_dir / "nudger_results.csv",
    index=False,
)

df_nudger.head()

Traitement batch 1/11 (100 ISBN)
Traitement batch 2/11 (100 ISBN)
Traitement batch 3/11 (100 ISBN)
Traitement batch 4/11 (100 ISBN)
Traitement batch 5/11 (100 ISBN)
Traitement batch 6/11 (100 ISBN)
Traitement batch 7/11 (100 ISBN)
Traitement batch 8/11 (100 ISBN)
Traitement batch 9/11 (100 ISBN)
Traitement batch 10/11 (100 ISBN)
Traitement batch 11/11 (36 ISBN)


,source,isbn_query,found,error,status_code,source_id,isbn,title,subtitle,authors,...,info_link,canonical_volume_link,industry_identifiers,url,format,offers_count,min_price,currency,last_updated,raw_data
0,nudger,9791038202733,True,None,None,9791038202733,9791038202733,#les mémés tome 1,NaN,[],...,https://nudger.fr/9791038202733,https://nudger.fr/9791038202733,[],https://nudger.fr/9791038202733,Album,5,7.37,EUR,1779343798487,"{'isbn': '9791038202733', 'title': '#les mémés..."
1,nudger,9782888906063,True,None,None,9782888906063,9782888906063,103ème escadrille de chasse,NaN,[],...,https://nudger.fr/9782888906063,https://nudger.fr/9782888906063,[],https://nudger.fr/9782888906063,Album,3,9.5,EUR,1779341894660,"{'isbn': '9782888906063', 'title': '103ème esc..."
2,nudger,9782344057001,True,None,None,9782344057001,9782344057001,2001 nights stories tome 1,NaN,[],...,https://nudger.fr/9782344057001,https://nudger.fr/9782344057001,[],https://nudger.fr/9782344057001,Album,2,32.0,EUR,1779340873092,"{'isbn': '9782344057001', 'title': '2001 night..."
3,nudger,9782845380790,True,None,None,9782845380790,9782845380790,20th century boys tome,NaN,[],...,https://nudger.fr/9782845380790,https://nudger.fr/9782845380790,[],https://nudger.fr/9782845380790,Non Precisé,2,5.26,EUR,1779340794455,"{'isbn': '9782845380790', 'title': '20th centu..."
4,nudger,9782809425659,True,None,None,9782809425659,9782809425659,20th Century Boys - Spin Off,NaN,[],...,https://nudger.fr/9782809425659,https://nudger.fr/9782809425659,[],https://nudger.fr/9782809425659,Tankobon,1,13.9,EUR,1779328820653,"{'isbn': '9782809425659', 'title': '20th Centu..."


In [17]:
df_google_books = pd.read_csv(
    output_dir / "google_books_results.csv",
    low_memory=False,
)

df_openlibrary = pd.read_csv(
    output_dir / "openlibrary_results.csv",
    low_memory=False,
)

df_bnf = pd.read_csv(
    output_dir / "bnf_results.csv",
    low_memory=False,
)

df_nudger = pd.read_csv(
    output_dir / "nudger_results.csv",
    low_memory=False,
)

In [19]:
print("Google Books :", len(df_google_books))
print("OpenLibrary :", len(df_openlibrary))
print("BNF :", len(df_bnf))
print("Nudger :", len(df_nudger))

Google Books : 1036
OpenLibrary : 1036
BNF : 1036
Nudger : 1036


In [20]:
for name, df in {
    "google_books": df_google_books,
    "openlibrary": df_openlibrary,
    "bnf": df_bnf,
    "nudger": df_nudger,
}.items():

    print(
        f"{name}: "
        f"{df['found'].sum()} / {len(df)}"
    )

google_books: 874 / 1036
openlibrary: 178 / 1036
bnf: 741 / 1036
nudger: 996 / 1036


In [21]:
coverage_df = pd.DataFrame(
    [
        ["Nudger", df_nudger["found"].mean() * 100],
        ["Google Books", df_google_books["found"].mean() * 100],
        ["BNF", df_bnf["found"].mean() * 100],
        ["OpenLibrary", df_openlibrary["found"].mean() * 100],
    ],
    columns=["source", "coverage_rate"],
)

coverage_df["coverage_rate"] = (
    coverage_df["coverage_rate"]
    .round(2)
)

coverage_df.sort_values(
    "coverage_rate",
    ascending=False,
)

,source,coverage_rate
0,Nudger,96.14
1,Google Books,84.36
2,BNF,71.53
3,OpenLibrary,17.18


## Couverture des sources

Le benchmark étendu confirme la très forte couverture de Nudger, qui retrouve près de la totalité des ouvrages de l'échantillon.

Google Books conserve une excellente couverture avec plus de 84 % des ISBN retrouvés, tandis que la BNF atteint plus de 71 %, ce qui représente une amélioration significative par rapport au benchmark initial.

OpenLibrary reste la source la plus limitée en matière de couverture avec environ 17 % des ISBN retrouvés. Son rôle semble davantage orienté vers l'enrichissement sémantique de certaines notices que vers une couverture exhaustive de la collection.

Ces résultats confirment que les différentes sources possèdent des rôles complémentaires dans la future stratégie multi-sources.

In [27]:
def compute_completeness(df, fields):
    found_df = df[df["found"] == True].copy()

    result = []

    for field in fields:
        if field not in found_df.columns:
            continue

        completion = (
            found_df[field]
            .notna()
            .mean()
            * 100
        )

        result.append(
            {
                "field": field,
                "completion_rate": round(completion, 2),
            }
        )

    return pd.DataFrame(result)

## Complétude des métadonnées Google Books

Google Books présente une excellente qualité de métadonnées sur les notices retrouvées.

Les champs essentiels tels que le titre, les auteurs, les catégories et la date de publication sont presque systématiquement renseignés. Le nombre de pages est également disponible pour la grande majorité des ouvrages.

Le champ description atteint un taux de complétude supérieur à 66 %, ce qui constitue un atout important pour les futurs systèmes de recommandation et le pipeline RAG.

Le principal point faible observé concerne l'éditeur, qui reste peu renseigné sur les ouvrages de l'échantillon.

Ces résultats confirment que Google Books constitue une source particulièrement intéressante pour la récupération des métadonnées générales et des contenus descriptifs.

In [23]:
google_fields = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "description",
    "categories",
    "page_count",
]

compute_completeness(
    df_google_books,
    google_fields,
)

,field,completion_rate
0,title,100.00
1,authors,100.00
2,publisher,15.79
3,published_date,99.66
4,description,66.02
5,categories,100.00
6,page_count,94.97


## Complétude des métadonnées Google Books

Google Books présente une excellente qualité de métadonnées sur les notices retrouvées.

Les champs essentiels tels que le titre, les auteurs, les catégories et la date de publication sont presque systématiquement renseignés. Le nombre de pages est également disponible pour la grande majorité des ouvrages.

Le champ description atteint un taux de complétude supérieur à 66 %, ce qui constitue un atout important pour les futurs systèmes de recommandation et le pipeline RAG.

Le principal point faible observé concerne l'éditeur, qui reste peu renseigné sur les ouvrages de l'échantillon.

Ces résultats confirment que Google Books constitue une source particulièrement intéressante pour la récupération des métadonnées générales et des contenus descriptifs.

In [24]:
compute_completeness(
    df_bnf,
    [
        "title",
        "authors",
        "publisher",
        "published_date",
        "description",
        "page_count",
        "bnf_ark",
    ],
)

,field,completion_rate
0,title,100.00
1,authors,100.00
2,publisher,99.46
3,published_date,98.52
4,description,99.46
5,page_count,73.95
6,bnf_ark,100.00


## Complétude des métadonnées BNF

La BNF présente une excellente qualité bibliographique sur les notices retrouvées.

Les champs titre, auteurs, éditeur, date de publication et description sont renseignés pour la quasi-totalité des ouvrages identifiés. L'identifiant ARK est systématiquement disponible, garantissant un lien fiable vers la notice bibliographique officielle.

L'extraction du nombre de pages atteint près de 74 % des notices trouvées, ce qui confirme l'intérêt de l'amélioration apportée lors de la phase de normalisation.

Ces résultats montrent que la BNF constitue une source particulièrement robuste pour les métadonnées bibliographiques françaises et les informations d'édition.

In [25]:
compute_completeness(
    df_openlibrary,
    [
        "title",
        "authors",
        "publisher",
        "published_date",
        "description",
        "subjects",
        "subject_people",
        "subject_places",
        "subject_times",
        "cover_large",
    ],
)

,field,completion_rate
0,title,100.00
1,authors,100.00
2,publisher,87.64
3,published_date,87.64
4,description,0.00
5,subjects,100.00
6,subject_people,100.00
7,subject_places,100.00
8,subject_times,100.00
9,cover_large,75.84


In [28]:
def compute_completeness_lists(df, fields):
    found_df = df[df["found"] == True].copy()

    results = []

    for field in fields:

        if field not in found_df.columns:
            continue

        completion = (
            found_df[field]
            .apply(
                lambda x: (
                    pd.notna(x)
                    and str(x) != "[]"
                    and str(x) != ""
                )
            )
            .mean()
            * 100
        )

        results.append(
            {
                "field": field,
                "completion_rate": round(completion, 2),
            }
        )

    return pd.DataFrame(results)

In [29]:
compute_completeness_lists(
    df_openlibrary,
    [
        "subjects",
        "subject_people",
        "subject_places",
        "subject_times",
        "cover_large",
    ]
)

,field,completion_rate
0,subjects,15.17
1,subject_people,1.12
2,subject_places,2.81
3,subject_times,1.12
4,cover_large,75.84


## Complétude des métadonnées OpenLibrary

OpenLibrary présente une couverture limitée sur l'échantillon étudié, avec seulement 17 % des ISBN retrouvés.

Les métadonnées bibliographiques classiques sont généralement bien renseignées lorsque l'ouvrage est identifié : titre, auteurs, éditeur et date de publication.

L'analyse détaillée des champs sémantiques montre cependant que leur présence reste relativement faible :

- subjects : 15 %
- subject_people : 1 %
- subject_places : 3 %
- subject_times : 1 %

Les couvertures constituent en revanche un point fort de la source, avec près de 76 % des notices retrouvées disposant d'une image exploitable.

Aucune description exploitable n'a été récupérée sur cet échantillon étendu.

Ces résultats montrent qu'OpenLibrary ne peut pas être considéré comme une source principale de métadonnées pour CollectionLens. Son intérêt réside principalement dans l'apport ponctuel de couvertures et de quelques informations thématiques complémentaires.

In [30]:
compute_completeness(
    df_nudger,
    [
        "title",
        "publisher",
        "categories",
        "page_count",
        "format",
        "offers_count",
        "min_price",
        "currency",
        "last_updated",
    ],
)

,field,completion_rate
0,title,100.00
1,publisher,92.27
2,categories,100.00
3,page_count,91.57
4,format,85.54
5,offers_count,100.00
6,min_price,82.63
7,currency,82.63
8,last_updated,100.00


## Complétude des métadonnées Nudger

Nudger présente la meilleure couverture parmi l'ensemble des sources évaluées avec plus de 96 % des ISBN retrouvés.

Les métadonnées principales sont très bien renseignées :

- titre : 100 %
- éditeur : 92 %
- nombre de pages : 92 %
- format : 86 %

Les informations commerciales sont également largement disponibles :

- nombre d'offres : 100 %
- prix minimum : 83 %
- devise : 83 %
- date de mise à jour : 100 %

Ces résultats confirment que Nudger constitue une source particulièrement adaptée aux univers BD, Manga et Comics. La qualité de sa couverture et la richesse de ses métadonnées en font une source de premier plan pour CollectionLens.

## Comparaison finale des sources

In [33]:
comparison_df = pd.DataFrame(
    [
        {
            "source": "Nudger",
            "coverage": 96.14,
            "description": "Faible",
            "publisher": "Excellent",
            "page_count": "Excellent",
            "categories": "Excellent",
        },
        {
            "source": "Google Books",
            "coverage": 84.36,
            "description": "Excellent",
            "publisher": "Faible",
            "page_count": "Excellent",
            "categories": "Correct",
        },
        {
            "source": "BNF",
            "coverage": 71.53,
            "description": "Excellent",
            "publisher": "Excellent",
            "page_count": "Bon",
            "categories": "Faible",
        },
        {
            "source": "OpenLibrary",
            "coverage": 17.18,
            "description": "Très faible",
            "publisher": "Bon",
            "page_count": "Faible",
            "categories": "Faible",
        },
    ]
)

comparison_df

,source,coverage,description,publisher,page_count,categories
0,Nudger,96.14,Faible,Excellent,Excellent,Excellent
1,Google Books,84.36,Excellent,Faible,Excellent,Correct
2,BNF,71.53,Excellent,Excellent,Bon,Faible
3,OpenLibrary,17.18,Très faible,Bon,Faible,Faible


## Conclusion générale

Le benchmark étendu réalisé sur 1036 séries représentatives de la collection permet de confirmer les tendances observées lors du benchmark intermédiaire.

Nudger apparaît comme la source principale grâce à sa couverture exceptionnelle et à la richesse de ses métadonnées spécialisées pour les univers BD, Manga et Comics.

Google Books constitue la meilleure source d'enrichissement textuel grâce à la qualité de ses descriptions, de ses auteurs et de ses informations bibliographiques générales.

La BNF apporte une forte valeur ajoutée pour la validation bibliographique et la récupération de métadonnées d'édition fiables.

OpenLibrary présente un intérêt plus limité dans le contexte de CollectionLens et sera utilisé uniquement comme source secondaire d'enrichissement lorsque des informations complémentaires ou des couvertures sont disponibles.

Le benchmark confirme la pertinence d'une future stratégie multi-sources reposant principalement sur Nudger, Google Books et BNF.

## Recommandations

À l'issue de ce benchmark, les prochaines étapes du projet sont :

1. Définir la stratégie d'agrégation multi-sources.
2. Définir le modèle cible des métadonnées CollectionLens.
3. Développer le pipeline d'agrégation.
4. Construire un dataset maître enrichi.
5. Préparer les futurs systèmes de recommandation.
6. Préparer le futur pipeline RAG.

In [34]:
coverage_df.to_csv(
    output_dir / "coverage_summary.csv",
    index=False,
)

comparison_df.to_csv(
    output_dir / "sources_comparison.csv",
    index=False,
)

print("Exports de synthèse générés.")

Exports de synthèse générés.
